# Imports

In [1]:
import os
import glob
import pandas as pd

BASE_DIR = "data_fake_bigdata"
FATO_DIR = os.path.join(BASE_DIR, "fato_vendas")

pd.set_option("display.max_columns", 50)

# Carregar Dimensões

In [2]:
dim_clientes = pd.read_parquet(os.path.join(BASE_DIR, "dim_clientes.parquet"))
dim_produtos = pd.read_parquet(os.path.join(BASE_DIR, "dim_produtos.parquet"))
dim_lojas = pd.read_parquet(os.path.join(BASE_DIR, "dim_lojas.parquet"))

print("Clientes:", dim_clientes.shape)
print("Produtos:", dim_produtos.shape)
print("Lojas:", dim_lojas.shape)

Clientes: (1000000, 7)
Produtos: (50000, 5)
Lojas: (5000, 5)


# Ver estrutura e tipos

In [3]:
dim_clientes.info()
dim_produtos.info()
dim_lojas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 7 columns):
 #   Column           Non-Null Count    Dtype         
---  ------           --------------    -----         
 0   id_cliente       1000000 non-null  int64         
 1   nome             1000000 non-null  object        
 2   cpf              1000000 non-null  object        
 3   cidade           1000000 non-null  object        
 4   estado           1000000 non-null  object        
 5   data_nascimento  1000000 non-null  datetime64[ns]
 6   data_criacao     1000000 non-null  datetime64[ns]
dtypes: datetime64[ns](2), int64(1), object(4)
memory usage: 53.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   id_produto      50000 non-null  int64         
 1   nome_produto    50000 non-null  object        
 2   cat

# Carregar fato (modo simples — tudo)

In [6]:
arquivos_fato = glob.glob(os.path.join(FATO_DIR, "*.parquet"))

fato_vendas = pd.concat(
    [pd.read_parquet(f) for f in arquivos_fato],
    ignore_index=True
)

print("Fato:", fato_vendas.shape)
fato_vendas.head()

Fato: (50000000, 10)


,id_venda,id_cliente,id_produto,id_loja,data_venda,quantidade,valor_unitario,valor_total,ano,mes
0,7000001,495136,27524,1157,2024-10-31,2,1117.07,2145.48,2024,10
1,7000002,403702,41708,511,2022-09-18,1,79.81,67.74,2022,9
2,7000003,836735,28053,3631,2022-01-15,1,1525.01,1324.93,2022,1
3,7000004,459135,19582,3264,2023-08-12,1,480.55,517.43,2023,8
4,7000005,731137,755,1969,2021-06-02,3,939.71,2721.07,2021,6


In [7]:
fato_vendas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000000 entries, 0 to 49999999
Data columns (total 10 columns):
 #   Column          Dtype         
---  ------          -----         
 0   id_venda        int64         
 1   id_cliente      int64         
 2   id_produto      int64         
 3   id_loja         int64         
 4   data_venda      datetime64[ns]
 5   quantidade      int64         
 6   valor_unitario  float64       
 7   valor_total     float64       
 8   ano             int32         
 9   mes             int32         
dtypes: datetime64[ns](1), float64(2), int32(2), int64(5)
memory usage: 3.4 GB


# Carregar fato (modo eficiente — apenas colunas necessárias)

In [6]:
colunas_interesse = [
    "id_cliente",
    "id_produto",
    "data_venda",
    "valor_total",
    "ano",
    "mes"
]

arquivos_fato = glob.glob(os.path.join(FATO_DIR, "*.parquet"))

fato_vendas = pd.concat(
    [pd.read_parquet(f, columns=colunas_interesse) for f in arquivos_fato],
    ignore_index=True
)

print("Fato (colunas reduzidas):", fato_vendas.shape)

Fato (colunas reduzidas): (50000000, 6)


# Métricas rápidas

## Receita total

In [8]:
fato_vendas["valor_total"].sum()

np.float64(134488412265.38)

## Receita por ano

In [9]:
fato_vendas.groupby("ano")["valor_total"].sum().sort_index()

ano
2020    2.247826e+10
2021    2.239053e+10
2022    2.239718e+10
2023    2.238598e+10
2024    2.245957e+10
2025    2.237690e+10
Name: valor_total, dtype: float64

## Receita por categoria

In [10]:
fato_prod = fato_vendas.merge(
    dim_produtos[["id_produto", "categoria"]],
    on="id_produto",
    how="left"
)

fato_prod.groupby("categoria")["valor_total"].sum().sort_values(ascending=False)

categoria
Mercado        2.298828e+10
Esportes       2.272756e+10
Brinquedos     2.244877e+10
Eletrônicos    2.239867e+10
Roupas         2.223846e+10
Livros         2.168668e+10
Name: valor_total, dtype: float64

## Top 10 clientes por faturamento

In [11]:
top_clientes = (
    fato_vendas
    .groupby("id_cliente")["valor_total"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
)

top_clientes

id_cliente
725256    330615.05
531167    324991.47
198876    324956.05
101921    323304.52
686190    317957.06
170018    315293.25
577027    311813.63
628621    310194.96
114555    305208.52
452844    305183.58
Name: valor_total, dtype: float64

In [12]:
top_clientes = top_clientes.reset_index().merge(
    dim_clientes[["id_cliente", "nome"]],
    on="id_cliente"
)

top_clientes

,id_cliente,valor_total,nome
0,725256,330615.05,Renan Cavalcanti
1,531167,324991.47,Arthur Gabriel Cirino
2,198876,324956.05,Ana Carolina Melo
3,101921,323304.52,Cauê Sampaio
4,686190,317957.06,Luiz Felipe Machado
5,170018,315293.25,Fernando Cunha
6,577027,311813.63,Otto Cirino
7,628621,310194.96,Sr. Ravi Lucca Campos
8,114555,305208.52,Murilo Caldeira
9,452844,305183.58,Dr. Enzo Gabriel Caldeira


## Análise temporal

In [13]:
fato_vendas["ano_mes"] = (
    fato_vendas["ano"].astype(str)
    + "-"
    + fato_vendas["mes"].astype(str).str.zfill(2)
)

fato_vendas.groupby("ano_mes")["valor_total"].sum().sort_index()

ano_mes
2020-01    1.903456e+09
2020-02    1.777674e+09
2020-03    1.901062e+09
2020-04    1.843707e+09
2020-05    1.901527e+09
               ...     
2025-08    1.901177e+09
2025-09    1.844766e+09
2025-10    1.902722e+09
2025-11    1.841913e+09
2025-12    1.900984e+09
Name: valor_total, Length: 72, dtype: float64

In [14]:
dim_produtos

,id_produto,nome_produto,categoria,preco_unitario,data_criacao
0,1,Produto 1,Esportes,38.82,2025-06-14
1,2,Produto 2,Eletrônicos,1909.98,2024-04-19
2,3,Produto 3,Eletrônicos,1204.50,2023-08-04
3,4,Produto 4,Esportes,1477.63,2021-06-05
4,5,Produto 5,Livros,501.04,2024-07-09
...,...,...,...,...,...
49995,49996,Produto 49996,Livros,1474.56,2022-04-26
49996,49997,Produto 49997,Eletrônicos,1241.12,2021-10-28
49997,49998,Produto 49998,Mercado,486.77,2021-05-14
49998,49999,Produto 49999,Livros,902.04,2022-02-07


In [4]:
dim_lojas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   id_loja        5000 non-null   int64         
 1   nome_loja      5000 non-null   object        
 2   cidade         5000 non-null   object        
 3   estado         5000 non-null   object        
 4   data_abertura  5000 non-null   datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(3)
memory usage: 195.4+ KB


In [5]:
dim_lojas

,id_loja,nome_loja,cidade,estado,data_abertura
0,1,Loja 1,Rocha,RS,2023-11-02
1,2,Loja 2,Andrade do Amparo,PB,2018-06-04
2,3,Loja 3,Jesus,PA,2023-02-05
3,4,Loja 4,Montenegro,CE,2026-01-22
4,5,Loja 5,Santos,MG,2016-06-09
...,...,...,...,...,...
4995,4996,Loja 4996,Guerra do Oeste,RJ,2016-08-30
4996,4997,Loja 4997,Castro,RO,2016-06-21
4997,4998,Loja 4998,Siqueira das Pedras,AP,2022-06-21
4998,4999,Loja 4999,Moreira,SP,2025-11-19
